# Лабораторна робота 7. Завдання 2. Вплив стиснення зображень до 150×150

Ноутбук 7.2. Виконав: Бабин Богдан Олегович, група 443-4

**Перед запуском:** Среда выполнения → Сменить среду выполнения → **T4 GPU**, потім «Выполнить все».

**Що змінено відносно завдання 1** (п. 14–16): вхід 300×300 → **150×150**, вилучено 4-у та 5-у пари Conv2D + MaxPooling,
у генераторах `target_size=(150, 150)` і при розпізнаванні власних зображень теж 150×150. Решта — як у завданні 1.

> Оновлення до Keras 3: у візуалізації проміжних шарів `inputs = model.input` → `model.inputs`
> (у Keras 3 для Sequential `model.input` видає помилку). Останню комірку базового ноутбука `runtime.unassign()` прибрано —
> вона відключає середовище і всі файли зникають.

## Завантаження наборів даних

Навчальний набір `horse-or-human.zip` (з базового ноутбука):

In [ ]:
!wget -q https://storage.googleapis.com/learning-datasets/horse-or-human.zip

**п. 2. Додатковий набір даних для перевірки** `validation-horse-or-human.zip`:

In [ ]:
!wget -q https://storage.googleapis.com/learning-datasets/validation-horse-or-human.zip

**п. 3. Розпаковуємо набори в різні папки:** одну для навчання, другу для перевірки.

In [ ]:
import zipfile

# Unzip the dataset
local_zip = './horse-or-human.zip'
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('./horse-or-human')

local_zip = './validation-horse-or-human.zip'          # додано: перевірочний набір
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('./validation-horse-or-human')

zip_ref.close()

Кожна папка містить підкаталоги `horses` і `humans`. Мітки явно не задаються: `ImageDataGenerator`
автоматично маркує зображення за назвою підкаталогу.

In [ ]:
import os

# Directory with training horse pictures
train_horse_dir = os.path.join('./horse-or-human/horses')

# Directory with training human pictures
train_human_dir = os.path.join('./horse-or-human/humans')

# Directory with validation horse pictures                 (додано)
validation_horse_dir = os.path.join('./validation-horse-or-human/horses')

# Directory with validation human pictures                 (додано)
validation_human_dir = os.path.join('./validation-horse-or-human/humans')

**п. 4. Перевіряємо імена файлів у каталогах**

In [ ]:
train_horse_names = os.listdir(train_horse_dir)
print(f'TRAIN SET HORSES: {train_horse_names[:10]}')

train_human_names = os.listdir(train_human_dir)
print(f'TRAIN SET HUMANS: {train_human_names[:10]}')

validation_horse_hames = os.listdir(validation_horse_dir)
print(f'VAL SET HORSES: {validation_horse_hames[:10]}')

validation_human_names = os.listdir(validation_human_dir)
print(f'VAL SET HUMANS: {validation_human_names[:10]}')

In [ ]:
print(f'total training horse images: {len(os.listdir(train_horse_dir))}')
print(f'total training human images: {len(os.listdir(train_human_dir))}')
print(f'total validation horse images: {len(os.listdir(validation_horse_dir))}')
print(f'total validation human images: {len(os.listdir(validation_human_dir))}')

**п. 5. Показуємо кілька навчальних зображень**

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Parameters for our graph; we'll output images in a 4x4 configuration
nrows = 4
ncols = 4

# Index for iterating over images
pic_index = 0

In [ ]:
# Set up matplotlib fig, and size it to fit 4x4 pics
fig = plt.gcf()
fig.set_size_inches(ncols * 4, nrows * 4)

pic_index += 8
next_horse_pix = [os.path.join(train_horse_dir, fname)
                for fname in train_horse_names[pic_index-8:pic_index]]
next_human_pix = [os.path.join(train_human_dir, fname)
                for fname in train_human_names[pic_index-8:pic_index]]

for i, img_path in enumerate(next_horse_pix+next_human_pix):
  # Set up subplot; subplot indices start at 1
  sp = plt.subplot(nrows, ncols, i + 1)
  sp.axis('Off') # Don't show axes (or gridlines)

  img = mpimg.imread(img_path)
  plt.imshow(img)

plt.show()

## п. 6. Будуємо модель

In [ ]:
import tensorflow as tf
import time

Бінарна класифікація (2 класи), тому на виході **один нейрон з sigmoid**: значення 0..1 — імовірність класу 1.
`class_indices`: `horses` = 0, `humans` = 1. Вхід — 150×150×3.

In [ ]:
model = tf.keras.models.Sequential([
    # Note the input shape is the desired size of the image 150x150 with 3 bytes color
    # This is the first convolution
    tf.keras.layers.Conv2D(16, (3,3), activation='relu', input_shape=(150, 150, 3)),
    tf.keras.layers.MaxPooling2D(2, 2),
    # The second convolution
    tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    # The third convolution
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    # # The fourth convolution (вилучено в завданні 2)
    # tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    # tf.keras.layers.MaxPooling2D(2,2),
    # # The fifth convolution
    # tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    # tf.keras.layers.MaxPooling2D(2,2),
    # Flatten the results to feed into a DNN
    tf.keras.layers.Flatten(),
    # 512 neuron hidden layer
    tf.keras.layers.Dense(512, activation='relu'),
    # Only 1 output neuron. It will contain a value from 0-1 where 0 for 1 class ('horses') and 1 for the other ('humans')
    tf.keras.layers.Dense(1, activation='sigmoid')
])

**п. 7. Короткий опис моделі** — розміри: 148 → 74 → 72 → 36 → 34 → **17×17×64** (п. 15). Увага: у Dense тепер **більше** входів, ніж у завданні 1: 17·17·64 = 18 496 проти 7·7·64 = 3 136, бо вилучено дві пари згортка+MaxPooling, які зменшували карту ознак. Тому параметрів ~9,5 млн замість ~1,7 млн, хоча зображення менші.

In [ ]:
model.summary()

**п. 8. Компіляція з тими самими параметрами:** `binary_crossentropy`, RMSprop, learning_rate=0.001.

In [ ]:
from tensorflow.keras.optimizers import RMSprop

model.compile(loss='binary_crossentropy',
              optimizer=RMSprop(learning_rate=0.001),
              metrics=['accuracy'])

## п. 9. Генератори даних
Генератор читає зображення з папок, переводить у `float32`, нормалізує (`rescale=1/255`) і віддає пакетами разом з мітками.
**Додано другий генератор — для перевірочних даних**, він вказує на каталог `validation-horse-or-human`.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# All images will be rescaled by 1./255
train_datagen = ImageDataGenerator(rescale=1/255)
validation_datagen = ImageDataGenerator(rescale=1/255)       # додано

# Flow training images in batches of 128 using train_datagen generator
train_generator = train_datagen.flow_from_directory(
        './horse-or-human/',  # This is the source directory for training images
        target_size=(150, 150),  # All images will be resized to 150x150
        batch_size=128,
        # Since we use binary_crossentropy loss, we need binary labels
        class_mode='binary')

# Flow validation images in batches of 32 using validation_datagen generator      (додано)
validation_generator = validation_datagen.flow_from_directory(
        './validation-horse-or-human/',  # This is the source directory for validation images
        target_size=(150, 150),
        batch_size=32,
        class_mode='binary')

print('Класи навчального генератора:', train_generator.class_indices)
print('Класи перевірочного генератора:', validation_generator.class_indices, '| каталог:', validation_generator.directory)

## п. 10–11. Навчання з перевіркою

In [ ]:
start = time.time()
history = model.fit(
      train_generator,
      steps_per_epoch=8,
      epochs=15,
      verbose=1,
      validation_data = validation_generator,        # додано
      validation_steps=8)                            # додано
train_time = time.time() - start
print(f'Час навчання: {train_time:.0f} с')

## п. 12. Графіки історії навчання

In [ ]:
acc, val_acc = history.history['accuracy'], history.history['val_accuracy']
loss, val_loss = history.history['loss'], history.history['val_loss']
epochs = range(1, len(acc) + 1)

plt.figure(figsize=(13, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, acc, 'b', label='Навчальні дані')
plt.plot(epochs, val_acc, 'r', label='Перевірочні дані')
plt.title('Точність'); plt.xlabel('Епоха'); plt.legend(); plt.grid(alpha=.3)
plt.subplot(1, 2, 2)
plt.plot(epochs, loss, 'b', label='Навчальні дані')
plt.plot(epochs, val_loss, 'r', label='Перевірочні дані')
plt.title('Втрати'); plt.xlabel('Епоха'); plt.legend(); plt.grid(alpha=.3)
plt.suptitle('Вхід 150×150'); plt.tight_layout(); plt.show()

print(f'Остання епоха: train acc {acc[-1]*100:.1f}%, val acc {val_acc[-1]*100:.1f}%')
print(f'Найкраща val acc: {max(val_acc)*100:.1f}% (епоха {val_acc.index(max(val_acc)) + 1})')
print(f'Параметрів моделі: {model.count_params():,} | час навчання: {train_time:.0f} с')

In [ ]:
import pandas as pd
summary = pd.DataFrame([{'розмір входу': '150×150', 'параметрів': model.count_params(),
                         'train acc, %': round(acc[-1] * 100, 1), 'val acc, %': round(val_acc[-1] * 100, 1),
                         'найкраща val acc, %': round(max(val_acc) * 100, 1),
                         'train loss': round(loss[-1], 3), 'val loss': round(val_loss[-1], 3),
                         'час навчання, с': round(train_time)}])
summary.to_csv('lab7_7_2_results.csv', index=False)
summary

## п. 18. Розпізнавання власних зображень
Завантажте різні зображення: схожі на навчальні та «хитрі» — людина і кінь разом, людина з довгим волоссям,
жінка в довгій сукні, людина в капелюсі, янгол з крилами, коні в траві, білий кінь на траві, мультяшні та іграшкові коні.
**Використовуйте ті самі фото, що й у завданні 1**, щоб порівняти результати.

Вихід мережі > 0.5 → людина, < 0.5 → кінь. Чим ближче до 0 або 1, тим впевненіша відповідь.

In [ ]:
import numpy as np
from google.colab import files
from tensorflow.keras.utils import load_img, img_to_array

uploaded = files.upload()

results = []
for fn in uploaded.keys():

  # predicting images
  path = '/content/' + fn
  img = load_img(path, target_size=(150, 150))
  x = img_to_array(img)
  x /= 255
  x = np.expand_dims(x, axis=0)

  images = np.vstack([x])
  classes = model.predict(images, batch_size=10, verbose=0)
  print(classes[0])

  if classes[0]>0.5:
    print(fn + " is a human")
  else:
    print(fn + " is a horse")
  results.append((fn, float(classes[0][0])))

Ті самі результати у вигляді сітки з картинками (додано):

In [ ]:
cols = 4; rows_n = int(np.ceil(len(results) / cols))
plt.figure(figsize=(cols * 3.5, rows_n * 3.8))
for i, (fn, p) in enumerate(results):
    plt.subplot(rows_n, cols, i + 1)
    plt.imshow(load_img('/content/' + fn)); plt.axis('off')
    label = 'людина' if p > 0.5 else 'кінь'
    conf = p if p > 0.5 else 1 - p
    plt.title(f'{label} ({conf*100:.0f}%)\n{fn[:28]}', fontsize=9)
plt.suptitle('Розпізнавання власних зображень, вхід 150×150'); plt.tight_layout(); plt.show()

own = pd.DataFrame([{'файл': fn, 'вихід мережі': round(p, 4), 'відповідь': 'людина' if p > 0.5 else 'кінь',
                     'впевненість, %': round((p if p > 0.5 else 1 - p) * 100, 1)} for fn, p in results])
own.to_csv('lab7_7_2_own_images.csv', index=False)
own

## Візуалізація проміжних представлень
Що «бачить» мережа: кожен рядок — вихід шару, кожна картинка в рядку — один фільтр (карта ознак).

In [ ]:
import numpy as np
import random
from tensorflow.keras.utils import img_to_array, load_img

# Define a new Model that will take an image as input, and will output
# intermediate representations for all layers in the previous model after
# the first.
successive_outputs = [layer.output for layer in model.layers[1:]]
visualization_model = tf.keras.models.Model(inputs = model.inputs, outputs = successive_outputs)   # Keras 3: model.inputs

# Prepare a random input image from the training set.
horse_img_files = [os.path.join(train_horse_dir, f) for f in train_horse_names]
human_img_files = [os.path.join(train_human_dir, f) for f in train_human_names]
img_path = random.choice(horse_img_files + human_img_files)

img = load_img(img_path, target_size=(150, 150))  # this is a PIL image
x = img_to_array(img)  # Numpy array with shape (150, 150, 3)
x = x.reshape((1,) + x.shape)  # Numpy array with shape (1, 150, 150, 3)

# Scale by 1/255
x /= 255

# Run the image through the network, thus obtaining all
# intermediate representations for this image.
successive_feature_maps = visualization_model.predict(x)

# These are the names of the layers, so you can have them as part of the plot
layer_names = [layer.name for layer in model.layers[1:]]

# Display the representations
for layer_name, feature_map in zip(layer_names, successive_feature_maps):
  if len(feature_map.shape) == 4:

    # Just do this for the conv / maxpool layers, not the fully-connected layers
    n_features = feature_map.shape[-1]  # number of features in feature map

    # The feature map has shape (1, size, size, n_features)
    size = feature_map.shape[1]

    # Tile the images in this matrix
    display_grid = np.zeros((size, size * n_features))
    for i in range(n_features):
      x = feature_map[0, :, :, i]
      x -= x.mean()
      x /= (x.std() + 1e-8)
      x *= 64
      x += 128
      x = np.clip(x, 0, 255).astype('uint8')

      # Tile each filter into this big horizontal grid
      display_grid[:, i * size : (i + 1) * size] = x

    # Display the grid
    scale = 20. / n_features
    plt.figure(figsize=(scale * n_features, scale))
    plt.title(layer_name)
    plt.grid(False)
    plt.imshow(display_grid, aspect='auto', cmap='viridis')

In [ ]:
files.download('lab7_7_2_results.csv')
files.download('lab7_7_2_own_images.csv')

## Висновки (завдання 2)
*(заповніть після запуску)* Порівняння з завданням 1: точність на навчальних ..., на перевірочних ..., час навчання ...,
перенавчання ...; розпізнавання реальних зображень ...; що краще — менші чи більші зображення ...